# cjm-capability-primitives

> The dep-light primitives library for the cjm capability ecosystem — the shared cross-task result DTOs (data nouns) that tool capabilities emit and task adapters, workflow cores, and the composition layer all consume.

**Result nouns vs adapter machinery:** a single-task tool capability emits its task's typed data noun (e.g. `TranscriptionResult`) and depends on *only* that — the adapter (`TaskAdapter`, the required tool protocol, the cache/persistence helpers) lives in the per-task `cjm-<task>-adapter-interface` lib and is constructed *around* the tool, never imported *by* it. The capability-side counterpart to `cjm-context-graph-primitives`.

## Install

```bash
pip install cjm_capability_primitives
```

## Project Structure

```
nbs/
├── forced_alignment.ipynb  # Standardized word-level forced-alignment DTOs — the data noun forced-alignment tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
├── source_separation.ipynb # Standardized result DTO for the source-separation (audio-preprocessing) task — the data noun source-separation tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
├── transcription.ipynb     # Standardized result DTO for the transcription task — the data noun tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
└── vad.ipynb               # Standardized result DTO for the voice-activity-detection task — the data noun VAD tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.
```

Total: 4 notebooks

## Module Dependencies

```mermaid
graph LR
    forced_alignment["forced_alignment<br/>Forced Alignment Result"]
    source_separation["source_separation<br/>Source Separation Result"]
    transcription["transcription<br/>Transcription Result"]
    vad["vad<br/>VAD Result"]

```

No cross-module dependencies detected.

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### Forced Alignment Result (`forced_alignment.ipynb`)
> Standardized word-level forced-alignment DTOs — the data noun forced-alignment tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.forced_alignment import (
    ForcedAlignItem,
    ForcedAlignResult
)
```
#### Classes

```python
@dataclass
class ForcedAlignItem:
    "A single word-level alignment result."
    
    text: str  # The aligned word (punctuation typically stripped by model)
    start_time: float  # Start time in seconds
    end_time: float  # End time in seconds
```

```python
@dataclass
class ForcedAlignResult:
    "Standardized output for all forced alignment capabilities."
    
    items: List[ForcedAlignItem]  # Word-level alignments
    metadata: Dict[str, Any] = field(...)  # Capability-specific metadata
    
    def from_dict(
        "Reconstruct from a wire payload, re-typing nested items.

`items` holds typed `ForcedAlignItem` objects, so the substrate's typed
wire envelope (stage 2) reconstructs them host-side here rather than
leaving bare dicts (which would break attribute access like `it.text`)."
```


### Source Separation Result (`source_separation.ipynb`)
> Standardized result DTO for the source-separation (audio-preprocessing) task — the data noun source-separation tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.source_separation import (
    SourceSeparationResult
)
```
#### Classes

```python
@dataclass
class SourceSeparationResult:
    """
    Standardized output for source-separation (audio-preprocessing) capabilities.
    
    The payload is an AUDIO ARTIFACT, not inline data: `output_path` is the
    produced isolated-audio file (e.g. the vocals stem) the tool wrote to the
    location the adapter chose. `metadata` carries the stats the fused-era
    return dict held (duration, sample_rate, model, stems_available, and any
    extra-stem paths when the tool was asked to keep them).
    """
    
    output_path: str  # Path to the produced isolated-audio artifact (e.g. vocals stem)
    metadata: Dict[str, Any] = field(...)  # Stats (duration, sample_rate, model, stems_available, other_stems, ...)
```


### Transcription Result (`transcription.ipynb`)
> Standardized result DTO for the transcription task — the data noun tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.transcription import (
    TranscriptionResult
)
```
#### Classes

```python
@dataclass
class TranscriptionResult:
    "Standardized output for all transcription plugins."
    
    text: str  # The transcribed text
    confidence: Optional[float]  # Overall confidence (0.0 to 1.0)
    segments: Optional[List[Dict[str, Any]]]  # Timestamped segments
    metadata: Dict[str, Any] = field(...)  # Additional metadata
```


### VAD Result (`vad.ipynb`)
> Standardized result DTO for the voice-activity-detection task — the data noun VAD tool capabilities emit and task adapters / workflow cores consume, wire-registered so results cross the worker boundary typed.

#### Import

```python
from cjm_capability_primitives.vad import (
    TimeRange,
    VADResult
)
```
#### Classes

```python
@dataclass
class TimeRange:
    "A temporal segment within an audio source (the VAD speech/silence span)."
    
    start: float  # Start time in seconds
    end: float  # End time in seconds
    label: str = 'speech'  # Segment type (e.g. 'speech')
    confidence: Optional[float]  # Detection confidence (0.0 to 1.0)
    payload: Dict[str, Any] = field(...)  # Extra data (reserved)
    
    def to_dict(self) -> Dict[str, Any]:  # Serialized representation
        "Convert to dictionary for JSON serialization."
```

```python
@dataclass
class VADResult:
    "Standardized output for voice-activity-detection capabilities."
    
    ranges: List[TimeRange]  # Detected speech segments, sorted by start
    metadata: Dict[str, Any] = field(...)  # Global VAD stats (duration, sample_rate, total_speech, ...)
    
    def from_dict(
        "Reconstruct from a wire payload, re-typing nested TimeRanges.

`ranges` holds typed `TimeRange` objects, so the substrate's typed wire
envelope (stage 2) reconstructs them host-side here rather than leaving
bare dicts (which would break attribute access like `r.start`)."
```
